# Combinando validadores *before* e *after*

Vimos como definir validadores before e after.

Podemos definir quantos quisermos de cada, e também misturá-los.

Os before rodam primeiro (de baixo para cima), depois o validador do Pydantic, depois os after (de cima para baixo).

Usar before e after juntos pode ser bem prático.

Lembre o exemplo do datetime:

In [1]:
from datetime import datetime
from typing import Any

import pytz
from dateutil.parser import parse
from pydantic import BaseModel, field_validator, ValidationError


class Model(BaseModel):
    dt: datetime

    @field_validator("dt", mode="before")
    @classmethod
    def parse_datetime(cls, value: Any):
        if isinstance(value, str):
            try:
                return parse(value)
            except Exception as ex:
                raise ValueError(str(ex))
        return value

Temos um único validador que acrescenta uma interpretação customizada de strings e deixa todo o resto como está, para o Pydantic tratar
os outros tipos.

Na verdade, o Pydantic também aceita inteiros — trata como *epoch* (segundos desde 1970-01-01 UTC):

In [2]:
print(f"Model(dt=100_000): {Model(dt=100_000)!r}")

Model(dt=100_000): Model(dt=datetime.datetime(1970, 1, 2, 3, 46, 40, tzinfo=TzInfo(0)))


E com uma string:

In [3]:
print(f'Model(dt="2020/1/1 3pm"): {Model(dt="2020/1/1 3pm")!r}')

Model(dt="2020/1/1 3pm"): Model(dt=datetime.datetime(2020, 1, 1, 15, 0))


Com a string, temos um datetime *naive*; com o epoch, um datetime *aware*.

Falta consistência.

Suponha que também queremos converter todos os datetimes (*naive* ou *aware*) para UTC *aware* (como no exemplo da aula de validadores after).

Poderíamos tentar tratar tudo no validador before — mas aí teríamos de lidar com todos os tipos de dados que quiséssemos aceitar.

Em vez disso, tratamos só as strings no before, deixamos o Pydantic fazer seu trabalho e usamos um validador after (agora com garantia de
que o valor é um datetime) para transformá-lo em UTC *aware*.

Lembrando o validador `make_utc`:

In [4]:
class Model(BaseModel):
    dt: datetime

    @field_validator("dt")
    @classmethod
    def make_utc(cls, dt: datetime) -> datetime:
        if dt.tzinfo is None:
            dt = pytz.utc.localize(dt)
        else:
            dt = dt.astimezone(pytz.utc)
        return dt

Agora combinamos os dois validadores:

In [5]:
class Model(BaseModel):
    dt: datetime

    @field_validator("dt", mode="before")
    @classmethod
    def parse_datetime(cls, value: Any):
        if isinstance(value, str):
            try:
                return parse(value)
            except Exception as ex:
                raise ValueError(str(ex))
        return value

    @field_validator("dt")
    @classmethod
    def make_utc(cls, dt: datetime) -> datetime:
        if dt.tzinfo is None:
            dt = pytz.utc.localize(dt)
        else:
            dt = dt.astimezone(pytz.utc)
        return dt

Isso implementa exatamente o que queríamos:

In [6]:
print(f"Model(dt=100_000): {Model(dt=100_000)!r}")

Model(dt=100_000): Model(dt=datetime.datetime(1970, 1, 2, 3, 46, 40, tzinfo=<UTC>))


In [7]:
print(f'Model(dt="2020/1/1 3pm"): {Model(dt="2020/1/1 3pm")!r}')

Model(dt="2020/1/1 3pm"): Model(dt=datetime.datetime(2020, 1, 1, 15, 0, tzinfo=<UTC>))


In [8]:
eastern = pytz.timezone('US/Eastern')

print(f"Model(dt=eastern.localize(datetime(2020, 1, 1, 3, 0, 0))): {Model(dt=eastern.localize(datetime(2020, 1, 1, 3, 0, 0)))!r}")

Model(dt=eastern.localize(datetime(2020, 1, 1, 3, 0, 0))): Model(dt=datetime.datetime(2020, 1, 1, 8, 0, tzinfo=<UTC>))


Antes de terminar, a ilustração do que vimos nos slides sobre a ordem de execução com validadores before e after misturados.

In [9]:
class Model(BaseModel):
    number: int

    @field_validator("number")
    @classmethod
    def after_validator_1(cls, value):
        print("after_validator_1")
        return value

    @field_validator("number")
    @classmethod
    def after_validator_2(cls, value):
        print("after_validator_2")
        return value

    @field_validator("number", mode="before")
    @classmethod
    def before_validator_1(cls, value):
        print("before_validator_1")
        return value

    @field_validator("number")
    @classmethod
    def after_validator_3(cls, value):
        print("after_validator_3")
        return value

    @field_validator("number", mode="before")
    @classmethod
    def before_validator_2(cls, value):
        print("before_validator_2")
        return value

In [10]:
print(f"Model(number=10): {Model(number=10)!r}")

before_validator_2
before_validator_1
after_validator_1
after_validator_2
after_validator_3
Model(number=10): Model(number=10)
